# 1. Getting Started & Preprocessing

STP-Bench exposes one entry point, the `STPred` class, for the whole
benchmark workflow: preprocessing data, training models, evaluating them
internally/externally, running inference on new slides, and downstream
biological analysis on the predictions.

This notebook covers the two things every other notebook in this series
assumes you already know — constructing an `STPred` instance and
discovering/validating its configs — and then the first real workflow step:
turning a data config into patches, embeddings, and CV splits.


> **Prerequisites**
> - STP-Bench installed (`bash scripts/create_env.sh`, see the
>   [README](../README.md#installation)).
> - Benchmark data downloaded for the dataset(s) used below (see
>   [README — Benchmark Data](../README.md#benchmark-data)), or your own
>   dataset added following
>   [docs/guide.md — Adding a New Dataset](../docs/guide.md#adding-a-new-dataset).
> - Run this notebook from the repo root, or pass `repo_root=` explicitly to
>   `STPred(...)`.


## Constructing an `STPred` instance

`STPred(...)` never touches disk — it only records settings. Config files
under `config/data/` and `config/model/` are resolved lazily, the first time
a workflow method (`preprocess()`, `train()`, ...) actually runs.


In [ ]:
from stpbench import STPred

stp = STPred(
    models=["LinearProb"],        # a model config name, or a list of them
    repo_root=".",            # directory containing config/, logs/, ...
    gpu=1,
    gpu_id=0,
)


`models` can be a single model config name (`config/model/<name>.yaml`) or a
list — every workflow method below runs all of them unless overridden
per-call. See the table in
[docs/guide.md — Creating an STPred Instance](../docs/guide.md#creating-an-stpred-instance)
for every constructor parameter (`debug`, `dry_run`, `wandb`, `log_file`, ...).


## Discovering what's available

Before pointing `STPred` at a dataset/model, list what configs actually
exist on disk, and inspect one before committing to it.


In [ ]:
# Data configs available under config/data/
print(stp.list_data())

# Model configs available under config/model/ (discovery — NOT stp.models)
print(stp.list_available_models())

# What THIS instance was constructed with (just echoes stp.models back)
print(stp.list_models())


`list_models()` and `list_available_models()` are easy to confuse:
`list_models()` only echoes back whatever was passed to
`STPred(models=[...])`; `list_available_models()` is the real discovery view
of every model config under `config/model/`.


In [ ]:
stp.describe_data("ncche/xenium")
stp.describe_model("LinearProb")


These also work as classmethods without constructing an instance first —
handy for a quick look before deciding what to pass to `STPred(...)`:

```python
STPred.list_data(repo_root=".")
STPred.list_available_models(repo_root=".")
```


## Validating configuration before a heavy run

`check()` (alias `preflight()`) verifies config shape and expected on-disk
artifacts *before* committing GPU time to a potentially long
preprocess/train job — data paths, `ids.csv`, gene-set/CV-split files, and a
sample of patch/embedding files per model.


In [ ]:
report = stp.check(data="ncche/xenium", mode="train", strict=False)
report  # {"ok": bool, "missing": [...], "reports": {<model>: {...}}}


- `mode="train"` (default) checks everything, including gene-set/CV-split
  files.
- `mode="eval"` is the same set of checks, used before `evaluate_*()`.
- `mode="inference"` skips the gene-set/CV-split checks that don't apply to
  unlabeled prediction targets (see notebook 3, *Prediction on New Slides*).

Pass `strict=True` to raise `FileNotFoundError` immediately instead of
getting a report back — useful once you're past the exploratory stage and
just want a hard fail-fast gate at the top of a script.


In [ ]:
stp.check(data="ncche/xenium", strict=True)  # raises if anything required is missing


## Preprocessing

Once a config checks out, `stp.preprocess(data=...)` turns it into
everything the rest of the pipeline needs: extracted patches, aligned ST
expression, a gene panel, cross-validation fold assignments, and
pre-extracted patch embeddings — one deduplicated plan shared across every
model configured on `stp`. That means calling this once for the six
models below does **not** run patch-embedding extraction six times — if
multiple models end up needing the same patch encoder + feature type, that
embedding is computed once and reused. Only a model's own optional
`extra_preprocess` step (graph building, similarity matrices, ...) is
inherently model-specific and still runs once per model that declares one.


In [ ]:
stp = STPred(models=["LinearProb", "EGN", "BLEEP", "TRIPLEX", "DeepSpot", "StFlow"], repo_root=".")
stp.preprocess(data="ncche/xenium")


### What actually runs

| Step | What it does | Output |
|---|---|---|
| Raw preprocessing | Extracts patches + ST expression per sample | `<data_dir>/patches/`, `<data_dir>/st/` |
| Gene-set preparation | Selects the `num_genes`-sized target panel | `<meta_dir>/<gene_type>_<num_genes>genes.json` |
| Cross-validation splits | Assigns each sample to train/test per fold | `<meta_dir>/ids.csv` (`fold_0`, `fold_1`, ...) |
| Feature extraction | Pre-extracts patch embeddings per model's encoder | `<data_dir>/emb/<feature_type>/features_<model_name>/` |

Every step is skipped if its output already exists — safe to call
`preprocess()` again after adding a new model to `stp.models` and only the
new model's missing feature-extraction step will actually run.


> **`mode: stpbench` needs `ids.csv` prepared up front.** If the data
> config uses `preprocess.mode: stpbench` (the case for every dataset already
> under `config/data/`, including `ncche/xenium` above), `<meta_dir>/ids.csv`
> — a CSV with at least a `sample_id` column — must already exist before
> calling `preprocess()`. Unlike `mode: raw` (which generates `ids.csv` by
> scanning `input_dir` itself), `stpbench` mode never scans the download
> directory for you and raises `FileNotFoundError` with the exact expected
> path if it's missing. Every dataset in this repo already has its
> `ids.csv` committed under `input/<namespace>/<name>/ids.csv` for exactly
> this reason — when adding your own `stpbench`-mode dataset, create that
> file yourself first.
>
> **Neighbor-patch models are expensive on freshly-downloaded data.** If any
> configured model uses `feature_type: neighbor`/`all` (e.g. DeepSpot), its
> neighbor patches are not included in the STP-Bench HF download — the first
> `preprocess()` call re-opens every raw WSI to run tissue segmentation +
> neighbor tiling from scratch, which can take tens of minutes *per
> gigapixel slide* with no fine-grained progress output. For a quick first
> run against a new dataset, prefer a `feature_type: global`-only model
> (e.g. LinearProb) instead.


### Preprocessing for only some models

Useful when you've already extracted features for one model and are adding
another to the comparison.


In [ ]:
stp.preprocess(data="ncche/xenium", models=["LinearProb"])  # only this model's plan


### Forcing re-processing

`overwrite=True` re-runs every step from scratch, ignoring existing outputs.
This is expensive (patch extraction and feature extraction over every
sample) — reach for it only when the underlying raw data actually changed,
not to "be safe."


In [ ]:
stp.preprocess(data="ncche/xenium", overwrite=True)


### Dry-running a plan

Pass `dry_run=True` (per-call, or `STPred(..., dry_run=True)` for every call
on that instance) to get back the *computed* plan — what would run and why
— without executing anything. Useful for sanity-checking a new data config
before committing to a long job.


In [ ]:
plan = stp.preprocess(data="ncche/xenium", dry_run=True)
plan


### Extra keyword arguments

Anything else passed to `preprocess()` is merged into the run's own
`preprocess:` config block for this call only (e.g. overriding `platform=`
or `n_splits=` without editing the YAML file):


In [ ]:
stp.preprocess(data="ncche/xenium", n_splits=3)


### Verifying the output

Don't just trust a clean exit code — `preprocess()` can complete without
every sample actually succeeding (some steps log and skip a bad sample
rather than raising). Spot-check the artifacts directly:


In [ ]:
import pandas as pd
from pathlib import Path

data_cfg = stp.describe_data("ncche/xenium")
meta_dir = Path(data_cfg["DATA"]["meta_dir"])
data_dir = Path(data_cfg["DATA"]["data_dir"])

ids = pd.read_csv(meta_dir / "ids.csv")
print(ids.head())
print("fold columns:", [c for c in ids.columns if c.startswith("fold_")])

# every sample should have a patch file and an ST file
missing_patches = [s for s in ids["sample_id"] if not (data_dir / "patches" / f"{s}.h5").exists()]
missing_st = [s for s in ids["sample_id"] if not (data_dir / "st" / f"{s}.h5ad").exists()]
print("missing patches:", missing_patches)
print("missing st:", missing_st)


### Bringing your own dataset

Everything above assumes a data config already exists under `config/data/`.
To integrate a brand-new dataset (raw WSI + ST directories in HEST layout),
see [docs/guide.md — Adding a New Dataset](../docs/guide.md#adding-a-new-dataset)
— it covers the raw input layout, writing the data config
(`STPred.init_data_config(...)`), and this same `preprocess()` call as the
last step.
